In [1]:
### con FECHA ULTIMA ### Mejora --> 2 de febrero 2026


#!/usr/bin/env python3
"""
retry_momentum_failed.py

Reintento puntual de Momentum 12m-1m para tickers fallidos.
Seguro, secuencial y auditable.
"""

import os
import time
import logging
import requests
import pandas as pd
from datetime import datetime, date, timedelta
from dotenv import load_dotenv
import psycopg2

# ---------------------------
# CONFIG FECHA
# ---------------------------
# 👉 Setear solo si querés forzar un día puntual
FECHA_MANUAL = None
 #FECHA_MANUAL = date(2026, 4, 6)

# ---------------------------
# ENV
# ---------------------------
load_dotenv()

POSTGRES_DB = os.getenv("POSTGRES_DB")
POSTGRES_USER = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_HOST = os.getenv("POSTGRES_HOST", "localhost")
POSTGRES_PORT = os.getenv("POSTGRES_PORT", "5432")
API_KEY = os.getenv("FMP_API_KEY")

BASE_URL = "https://financialmodelingprep.com/stable/historical-price-eod/dividend-adjusted"

# ---------------------------
# Logging
# ---------------------------
LOG_DIR = "output_ingest"
os.makedirs(LOG_DIR, exist_ok=True)

LOG_FILE = f"{LOG_DIR}/retry_momentum_{date.today().isoformat()}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE),
        logging.StreamHandler()
    ]
)

logging.info("=== START retry_momentum_failed ===")

# ---------------------------
# DB
# ---------------------------
def get_conn():
    return psycopg2.connect(
        dbname=POSTGRES_DB,
        user=POSTGRES_USER,
        password=POSTGRES_PASSWORD,
        host=POSTGRES_HOST,
        port=POSTGRES_PORT
    )

# ---------------------------
# Obtener última fecha en logs
# ---------------------------
def obtener_ultima_fecha_logs(conn):
    with conn.cursor() as cur:
        cur.execute("""
            SELECT MAX(created_at)::date
            FROM infraestructura.update_logs
            WHERE table_name = 'multifactor_momentum'
        """)
        r = cur.fetchone()
        return r[0]

# ---------------------------
# Obtener tickers fallidos
# ---------------------------
def obtener_tickers_fallidos(conn, fecha):
    with conn.cursor() as cur:
        cur.execute("""
            SELECT DISTINCT ticker
            FROM infraestructura.update_logs
            WHERE table_name = 'multifactor_momentum'
              AND created_at >= %s
              AND created_at <  %s
              AND status != 'success'
        """, (fecha, fecha + timedelta(days=1)))
        return [r[0] for r in cur.fetchall()]

# ---------------------------
# Momentum logic
# ---------------------------
def fetch_momentum_12m_1m(ticker):
    url = f"{BASE_URL}?symbol={ticker}&apikey={API_KEY}"
    r = requests.get(url, timeout=30)

    if r.status_code != 200:
        return None

    data = r.json()
    if not data or len(data) < 260:
        return None

    df = pd.DataFrame(data)
    df["date"] = pd.to_datetime(df["date"])
    df.sort_values("date", inplace=True)

    df_monthly = df.resample("M", on="date").last()
    if len(df_monthly) < 13:
        return None

    price_1m = float(df_monthly["adjClose"].iloc[-2])
    price_12m = float(df_monthly["adjClose"].iloc[-13])

    if price_12m <= 0:
        return None

    momentum = (price_1m / price_12m) - 1

    return {
        "price_1m": price_1m,
        "price_12m": price_12m,
        "momentum": momentum,
        "source": "FMP_historical_price_eod_dividend_adjusted"
    }

# ---------------------------
# SQL
# ---------------------------
INSERT_SQL = """
INSERT INTO api_raw.multifactor_momentum (
    ticker, fecha_de_consulta,
    price_1m, price_12m,
    momentum_12m_1m,
    source
)
VALUES (
    %(ticker)s, %(fecha)s,
    %(price_1m)s, %(price_12m)s,
    %(momentum)s,
    %(source)s
)
ON CONFLICT (ticker, fecha_de_consulta)
DO UPDATE SET
    price_1m = EXCLUDED.price_1m,
    price_12m = EXCLUDED.price_12m,
    momentum_12m_1m = EXCLUDED.momentum_12m_1m,
    source = EXCLUDED.source,
    updated_at = CURRENT_TIMESTAMP;
"""

# ---------------------------
# MAIN
# ---------------------------
def main():
    conn = get_conn()

    # ---------------------------
    # Resolver fecha objetivo
    # ---------------------------
    if FECHA_MANUAL:
        fecha_objetivo = FECHA_MANUAL
        logging.info(f"Usando fecha manual: {fecha_objetivo}")
    else:
        fecha_objetivo = obtener_ultima_fecha_logs(conn)
        if not fecha_objetivo:
            logging.error("No se pudo determinar fecha desde update_logs")
            conn.close()
            return
        logging.info(f"Usando última fecha detectada en logs: {fecha_objetivo}")

    tickers = obtener_tickers_fallidos(conn, fecha_objetivo)
    total = len(tickers)

    logging.info(f"Tickers a reintentar: {total}")

    ok, fail = 0, 0

    with conn.cursor() as cur:
        for ticker in tickers:
            try:
                data = fetch_momentum_12m_1m(ticker)
                if not data:
                    fail += 1
                    logging.warning(f"{ticker} sigue sin datos")
                    continue

                payload = data.copy()
                payload["ticker"] = ticker
                payload["fecha"] = datetime.today().date()

                cur.execute(INSERT_SQL, payload)
                conn.commit()
                ok += 1
                logging.info(f"{ticker} OK")

            except Exception as e:
                conn.rollback()
                fail += 1
                logging.error(f"{ticker} error: {e}")

            time.sleep(0.4)

    conn.close()
    logging.info(f"FIN RETRY MOMENTUM | OK={ok} | FAIL={fail}")

# ---------------------------
if __name__ == "__main__":
    main()


2026-04-06 20:53:15,266 | INFO | === START retry_momentum_failed ===
2026-04-06 20:53:15,397 | INFO | Usando última fecha detectada en logs: 2026-04-06
2026-04-06 20:53:15,402 | INFO | Tickers a reintentar: 399
2026-04-06 20:53:16,137 | WARNING | AACB sigue sin datos
2026-04-06 20:53:17,073 | INFO | AAL OK
2026-04-06 20:53:18,397 | INFO | ABT OK
2026-04-06 20:53:19,715 | INFO | ACU OK
2026-04-06 20:53:21,062 | INFO | ADI OK
2026-04-06 20:53:22,394 | INFO | ADM OK
2026-04-06 20:53:23,706 | INFO | ADP OK
2026-04-06 20:53:25,045 | INFO | AEP OK
2026-04-06 20:53:26,366 | INFO | AFL OK
2026-04-06 20:53:27,668 | INFO | AGYS OK
2026-04-06 20:53:28,981 | INFO | AIG OK
2026-04-06 20:53:30,108 | WARNING | AII sigue sin datos
2026-04-06 20:53:31,011 | INFO | AIR OK
2026-04-06 20:53:32,117 | WARNING | AIRO sigue sin datos
2026-04-06 20:53:33,037 | INFO | ALCO OK
2026-04-06 20:53:34,359 | INFO | ALE OK
2026-04-06 20:53:35,450 | WARNING | ALH sigue sin datos
2026-04-06 20:53:36,339 | INFO | ALLY OK


2026-04-06 20:56:42,219 | INFO | GIS OK
2026-04-06 20:56:43,325 | WARNING | GIW sigue sin datos
2026-04-06 20:56:44,223 | INFO | GLW OK
2026-04-06 20:56:45,328 | WARNING | GLXY sigue sin datos
2026-04-06 20:56:46,204 | INFO | GPC OK
2026-04-06 20:56:47,497 | INFO | GPJA OK
2026-04-06 20:56:48,788 | INFO | GRC OK
2026-04-06 20:56:49,894 | WARNING | GSRF sigue sin datos
2026-04-06 20:56:50,847 | INFO | GT OK
2026-04-06 20:56:51,977 | WARNING | GTEN sigue sin datos
2026-04-06 20:56:52,920 | INFO | GTN OK
2026-04-06 20:56:54,230 | INFO | HAL OK
2026-04-06 20:56:55,546 | INFO | HAS OK
2026-04-06 20:56:56,859 | INFO | HBAN OK
2026-04-06 20:56:58,160 | INFO | HEI OK
2026-04-06 20:56:59,261 | WARNING | HNGE sigue sin datos
2026-04-06 20:57:00,249 | INFO | HNI OK
2026-04-06 20:57:01,574 | INFO | HP OK
2026-04-06 20:57:02,922 | INFO | HPQ OK
2026-04-06 20:57:04,237 | INFO | HRB OK
2026-04-06 20:57:05,547 | INFO | HRL OK
2026-04-06 20:57:06,846 | INFO | HSY OK
2026-04-06 20:57:07,943 | WARNING | 

2026-04-06 21:00:10,999 | WARNING | TACH sigue sin datos
2026-04-06 21:00:11,708 | WARNING | TACO sigue sin datos
2026-04-06 21:00:12,635 | INFO | TAP OK
2026-04-06 21:00:14,140 | INFO | TGNA OK
2026-04-06 21:00:15,443 | INFO | TGT OK
2026-04-06 21:00:16,757 | INFO | THC OK
2026-04-06 21:00:18,126 | INFO | TPC OK
2026-04-06 21:00:19,441 | INFO | TRAK OK
2026-04-06 21:00:20,724 | INFO | TRMK OK
2026-04-06 21:00:21,834 | WARNING | TVA sigue sin datos
2026-04-06 21:00:22,538 | WARNING | TVAI sigue sin datos
2026-04-06 21:00:23,408 | INFO | UDR OK
2026-04-06 21:00:24,703 | INFO | UHAL OK
2026-04-06 21:00:26,001 | INFO | UNM OK
2026-04-06 21:00:27,313 | INFO | USAU OK
2026-04-06 21:00:28,629 | INFO | USB OK
2026-04-06 21:00:29,921 | INFO | USLM OK
2026-04-06 21:00:31,037 | WARNING | VACI sigue sin datos
2026-04-06 21:00:31,978 | INFO | VHI OK
2026-04-06 21:00:33,082 | WARNING | VIA sigue sin datos
2026-04-06 21:00:33,775 | WARNING | VNME sigue sin datos
2026-04-06 21:00:34,498 | WARNING | V